In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change Questionnnair -> Questionnaire
# if your folder uses the correct spelling.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "BPQ_I.xpt"
csv_file = DATA_DIR / "BPQ_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 BPQ_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BPQ020",
    "BPQ030",
    "BPD035",
    "BPQ040A",
    "BPQ050A",
    "BPQ080",
    "BPQ060",
    "BPQ070",
    "BPQ090D",
    "BPQ100D"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BPQ_I columns/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6327:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 11:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,327 rows x 11 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached BPQ_I:
#
#     genuine decimal fields = NONE
#     string fields          = NONE
#
#
# All 11 variables contain only whole-number values:
#
#     SEQN      -> respondent ID
#
#     BPQ...    -> questionnaire response codes
#
#     BPD035    -> age in whole years / special codes
#
#
# Therefore all 11 fields can safely become nullable Int64.
#
#
# IMPORTANT:
#
# This applies ONLY to BPQ_I.
#
# Your broader fidelity rule remains:
#
#     genuine Dietary decimals
#     genuine Laboratory decimals
#     Examination measurements
#     sample weights
#
# remain AS IS and must NOT be globally rounded.
#

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
#
# The attached BPQ_I contains NONE.
#
# Therefore:
#
#     DO NOT replace anything with zero.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 11 BPQ_I fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in BPQ_I."
    )


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary zero values "
        "found in BPQ_I."
    )


print(
    "PASS: BPQ_I contains 0 tiny artifacts "
    "and 0 ordinary zeros."
)


# ============================================================
# 6. NO ZERO RESTORATION REQUIRED
# ============================================================

#
# Since there is no tiny-zero corruption,
# the original XPT is directly our fidelity reference.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero restoration required."
)


# ============================================================
# 7. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() below is ONLY used as a diagnostic comparison.
#
# It does NOT modify source values.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal variables "
        "found in BPQ_I.\n\n"
        "These values will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "\nPASS: No genuine decimal variables "
    "found in BPQ_I."
)


# ============================================================
# 8. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 9. VALIDATE ALL VARIABLES ARE WHOLE-NUMBER
# ============================================================

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: All 11 BPQ_I variables are "
    "whole-number/code fields."
)


# ============================================================
# 10. CONVERT ALL FIELDS TO NULLABLE Int64
# ============================================================

#
# NO round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 11 BPQ_I fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 11. HELPER - FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - BPQ020
# ============================================================

#
# Ever told had high blood pressure:
#
#     1 = Yes          2,090
#     2 = No           4,231
#     7 = Refused          0
#     9 = Don't know       6
#
# Missing = 0
#

check_frequency(
    "BPQ020",
    {
        1: 2090,
        2: 4231,
        9: 6
    },
    0
)


# ============================================================
# 13. CDC CHECK - BPQ030
# ============================================================

#
# Told had high blood pressure on 2+ visits:
#
#     1 = Yes          1,659
#     2 = No             417
#     7 = Refused          1
#     9 = Don't know      13
#
# Missing = 4,237
#

check_frequency(
    "BPQ030",
    {
        1: 1659,
        2: 417,
        7: 1,
        9: 13
    },
    4237
)


# ============================================================
# 14. BPQ020 -> BPQ030 ROUTING CHECK
# ============================================================

#
# BPQ030 should be populated exactly when:
#
#     BPQ020 = 1
#

bpq020_yes = (
    df["BPQ020"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq030_present = (
    df["BPQ030"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq030_routing_mismatch = int(
    np.sum(
        bpq020_yes
        !=
        bpq030_present
    )
)


print(
    "\n--- BPQ020 / BPQ030 ROUTING ---"
)

print(
    "Routing mismatches:",
    bpq030_routing_mismatch
)


if bpq030_routing_mismatch != 0:

    raise ValueError(
        "BPQ030 availability does not match BPQ020=1."
    )


print(
    "PASS: BPQ020/BPQ030 routing validated."
)


# ============================================================
# 15. CDC CHECK - BPD035
# ============================================================

#
# Age first told had hypertension:
#
#     7-79  = 1,996
#     80    =    32
#     777   =     0
#     999   =    61
#
# Missing = 4,238
#

age_7_79 = int(
    df["BPD035"]
    .between(
        7,
        79
    )
    .sum()
)


age_80 = int(
    df["BPD035"]
    .eq(80)
    .sum()
)


age_777 = int(
    df["BPD035"]
    .eq(777)
    .sum()
)


age_999 = int(
    df["BPD035"]
    .eq(999)
    .sum()
)


age_missing = int(
    df["BPD035"]
    .isna()
    .sum()
)


print(
    "\n--- BPD035 ---"
)

print(
    "Age 7-79:",
    f"{age_7_79:,}"
)

print(
    "80 or older:",
    age_80
)

print(
    "777:",
    age_777
)

print(
    "999:",
    age_999
)

print(
    "Missing:",
    f"{age_missing:,}"
)


if age_7_79 != 1996:

    raise ValueError(
        "BPD035 age 7-79 count differs."
    )


if age_80 != 32:

    raise ValueError(
        "BPD035 age 80+ count differs."
    )


if age_777 != 0:

    raise ValueError(
        "Unexpected BPD035 refused values."
    )


if age_999 != 61:

    raise ValueError(
        "BPD035 don't-know count differs."
    )


if age_missing != 4238:

    raise ValueError(
        "BPD035 missing count differs."
    )


print(
    "PASS: BPD035 matches CDC."
)


# ============================================================
# 16. BPD035 AVAILABILITY CHECK
# ============================================================

#
# BPQ020 = 1:
#
#     total = 2,090
#
# BPD035 populated among them:
#
#     2,089
#
# BPD035 missing among them:
#
#     1
#
# No BPD035 value appears outside BPQ020=1.
#

bpq020_yes_series = (
    df["BPQ020"]
    .eq(1)
    .fillna(False)
)


age_present_yes = int(
    (
        bpq020_yes_series
        &
        df["BPD035"].notna()
    )
    .sum()
)


age_missing_yes = int(
    (
        bpq020_yes_series
        &
        df["BPD035"].isna()
    )
    .sum()
)


age_outside_yes = int(
    (
        ~bpq020_yes_series
        &
        df["BPD035"].notna()
    )
    .sum()
)


print(
    "\n--- BPD035 ROUTING ---"
)

print(
    "BPQ020=1 with BPD035 populated:",
    age_present_yes
)

print(
    "BPQ020=1 with BPD035 missing:",
    age_missing_yes
)

print(
    "BPD035 populated outside BPQ020=1:",
    age_outside_yes
)


if age_present_yes != 2089:

    raise ValueError(
        "Unexpected BPD035 populated count."
    )


if age_missing_yes != 1:

    raise ValueError(
        "Unexpected BPD035 missing count among BPQ020=1."
    )


if age_outside_yes != 0:

    raise ValueError(
        "BPD035 unexpectedly populated outside BPQ020=1."
    )


print(
    "PASS: BPD035 availability pattern validated."
)


# ============================================================
# 17. CDC CHECK - BPQ040A
# ============================================================

#
# Ever told to take prescribed medicine for hypertension:
#
#     1 = Yes          1,851
#     2 = No             238
#     9 = Don't know       1
#
# Missing = 4,237
#

check_frequency(
    "BPQ040A",
    {
        1: 1851,
        2: 238,
        9: 1
    },
    4237
)


# ============================================================
# 18. BPQ020 -> BPQ040A ROUTING
# ============================================================

bpq040_present = (
    df["BPQ040A"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq040_mismatch = int(
    np.sum(
        bpq020_yes
        !=
        bpq040_present
    )
)


if bpq040_mismatch != 0:

    raise ValueError(
        "BPQ040A availability does not match BPQ020=1."
    )


print(
    "PASS: BPQ020/BPQ040A routing validated."
)


# ============================================================
# 19. CDC CHECK - BPQ050A
# ============================================================

#
# Now taking prescribed medicine for HBP:
#
#     1 = Yes          1,567
#     2 = No             284
#
# Missing = 4,476
#

check_frequency(
    "BPQ050A",
    {
        1: 1567,
        2: 284
    },
    4476
)


# ============================================================
# 20. BPQ040A -> BPQ050A ROUTING
# ============================================================

bpq040_yes = (
    df["BPQ040A"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq050_present = (
    df["BPQ050A"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq050_mismatch = int(
    np.sum(
        bpq040_yes
        !=
        bpq050_present
    )
)


if bpq050_mismatch != 0:

    raise ValueError(
        "BPQ050A availability does not match BPQ040A=1."
    )


print(
    "PASS: BPQ040A/BPQ050A routing validated."
)


# ============================================================
# 21. CDC CHECK - BPQ080
# ============================================================

#
# Ever told cholesterol was high:
#
#     1 = Yes          1,960
#     2 = No           4,323
#     9 = Don't know      44
#
# Missing = 0
#

check_frequency(
    "BPQ080",
    {
        1: 1960,
        2: 4323,
        9: 44
    },
    0
)


# ============================================================
# 22. CDC CHECK - BPQ060
# ============================================================

#
# Ever had cholesterol checked:
#
#     1 = Yes          2,678
#     2 = No           1,499
#     7 = Refused          1
#     9 = Don't know     189
#
# Missing = 1,960
#
#
# BPQ080=1 skips BPQ060.
#

check_frequency(
    "BPQ060",
    {
        1: 2678,
        2: 1499,
        7: 1,
        9: 189
    },
    1960
)


# ============================================================
# 23. BPQ080 -> BPQ060 ROUTING
# ============================================================

#
# BPQ060 is asked when BPQ080 is:
#
#     2
#     7
#     9
#
# BPQ080=1 skips to BPQ070.
#

bpq060_expected = (
    df["BPQ080"]
    .isin(
        [2, 7, 9]
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq060_present = (
    df["BPQ060"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq060_mismatch = int(
    np.sum(
        bpq060_expected
        !=
        bpq060_present
    )
)


if bpq060_mismatch != 0:

    raise ValueError(
        "BPQ060 availability does not match "
        "BPQ080 routing."
    )


print(
    "PASS: BPQ080/BPQ060 routing validated."
)


# ============================================================
# 24. CDC CHECK - BPQ070
# ============================================================

#
# When cholesterol last checked:
#
#     1 = <1 year                 2,856
#     2 = 1-<2 years               870
#     3 = 2-<5 years               590
#     4 = 5 years or more           282
#     9 = Don't know                40
#
# Missing = 1,689
#

check_frequency(
    "BPQ070",
    {
        1: 2856,
        2: 870,
        3: 590,
        4: 282,
        9: 40
    },
    1689
)


# ============================================================
# 25. CDC CHECK - BPQ090D
# ============================================================

#
# Told to take prescribed medicine for cholesterol:
#
#     1 = Yes          1,524
#     2 = No           3,096
#     9 = Don't know      18
#
# Missing = 1,689
#

check_frequency(
    "BPQ090D",
    {
        1: 1524,
        2: 3096,
        9: 18
    },
    1689
)


# ============================================================
# 26. CHOLESTEROL ROUTING CHECK
# ============================================================

#
# BPQ070 and BPQ090D are populated when either:
#
#     BPQ080 = 1
#
# OR
#
#     BPQ060 = 1
#

chol_followup_expected = (
    (
        df["BPQ080"].eq(1)
        |
        df["BPQ060"].eq(1)
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq070_present = (
    df["BPQ070"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq090_present = (
    df["BPQ090D"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq070_mismatch = int(
    np.sum(
        chol_followup_expected
        !=
        bpq070_present
    )
)


bpq090_mismatch = int(
    np.sum(
        chol_followup_expected
        !=
        bpq090_present
    )
)


if bpq070_mismatch != 0:

    raise ValueError(
        "BPQ070 cholesterol routing differs."
    )


if bpq090_mismatch != 0:

    raise ValueError(
        "BPQ090D cholesterol routing differs."
    )


print(
    "PASS: BPQ070/BPQ090D cholesterol routing "
    "validated exactly."
)


# ============================================================
# 27. CDC CHECK - BPQ100D
# ============================================================

#
# Now taking prescribed cholesterol medicine:
#
#     1 = Yes          1,099
#     2 = No             424
#     9 = Don't know       1
#
# Missing = 4,803
#

check_frequency(
    "BPQ100D",
    {
        1: 1099,
        2: 424,
        9: 1
    },
    4803
)


# ============================================================
# 28. BPQ090D -> BPQ100D ROUTING
# ============================================================

#
# BPQ100D should exist exactly when:
#
#     BPQ090D = 1
#

bpq090_yes = (
    df["BPQ090D"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


bpq100_present = (
    df["BPQ100D"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


bpq100_mismatch = int(
    np.sum(
        bpq090_yes
        !=
        bpq100_present
    )
)


if bpq100_mismatch != 0:

    raise ValueError(
        "BPQ100D availability does not match BPQ090D=1."
    )


print(
    "PASS: BPQ090D/BPQ100D routing validated."
)


# ============================================================
# 29. VALID RESPONSE-CODE CHECK
# ============================================================

VALID_CODES = {
    "BPQ020": {1, 2, 9},
    "BPQ030": {1, 2, 7, 9},
    "BPQ040A": {1, 2, 9},
    "BPQ050A": {1, 2},
    "BPQ080": {1, 2, 9},
    "BPQ060": {1, 2, 7, 9},
    "BPQ070": {1, 2, 3, 4, 9},
    "BPQ090D": {1, 2, 9},
    "BPQ100D": {1, 2, 9}
}


print(
    "\n--- VALID RESPONSE CODE CHECK ---"
)


for col, allowed in VALID_CODES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed
        -
        allowed
    )


    print(
        f"{col}:",
        sorted(observed)
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected response codes: "
            f"{sorted(unexpected)}"
        )


print(
    "PASS: No unexpected response codes."
)


# ============================================================
# 30. CONFIRM NO ZEROS CREATED
# ============================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK BEFORE EXPORT ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values created:\n"
        + str(
            final_zero_counts
        )
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 31. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# BPQ_I required NO numeric correction.
#
# Therefore every numeric value must remain identical
# to the source XPT.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 11 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 32. EXPORT CSV
# ============================================================

#
# BPQ_I contains no genuine decimal variables.
#
# Still follow the standard fidelity export:
#
#     NO df.round(...)
#     NO float_format=
#
# This keeps the workflow consistent with mixed files
# containing genuine decimals.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 33. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 34. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 11 BPQ_I variables."
)


# ============================================================
# 35. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 36. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 37. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 38. RAW CSV .0 CHECK
# ============================================================

#
# BPQ_I contains no genuine decimal variables.
#
# Therefore checking ALL 11 fields for ".0" is safe
# for this specific file.
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any BPQ_I field."
)


# ============================================================
# 39. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found in final CSV:\n"
        + str(
            csv_zero_counts
        )
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 40. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 41. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 42. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "BPQ_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - "
    "Blood Pressure & Cholesterol"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Genuine decimal fields:",
    "NONE"
)


print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


print(
    "Ever told high blood pressure:",
    int(
        df["BPQ020"]
        .eq(1)
        .sum()
    )
)


print(
    "Currently taking HBP medicine:",
    int(
        df["BPQ050A"]
        .eq(1)
        .sum()
    )
)


print(
    "Ever told high cholesterol:",
    int(
        df["BPQ080"]
        .eq(1)
        .sum()
    )
)


print(
    "Currently taking cholesterol medicine:",
    int(
        df["BPQ100D"]
        .eq(1)
        .sum()
    )
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "BPQ_I contains no genuine decimal variables."
)


print(
    "All 11 fields can safely be exported "
    "as whole-number IDs/codes."
)


print(
    "Your broader NHANES fidelity rule remains unchanged:"
)


print(
    "Genuine Dietary and other continuous decimal "
    "variables remain exactly AS IS."
)

NHANES 2015-2016 BPQ_I XPT -> CSV
Rows: 6,327
Columns: 11
PASS: Structure = 6,327 rows x 11 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
Whole-number/code fields: ['SEQN', 'BPQ020', 'BPQ030', 'BPD035', 'BPQ040A', 'BPQ050A', 'BPQ080', 'BPQ060', 'BPQ070', 'BPQ090D', 'BPQ100D']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: BPQ_I contains 0 tiny artifacts and 0 ordinary zeros.

PASS: No tiny-to-zero restoration required.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
BPQ020: 0 fractional observations
BPQ030: 0 fractional observations
BPD035: 0 fractional observations
BPQ040A: 0 fractional observations
BPQ050A: 0 fractional observations
BPQ080: 0 fractional observations
BPQ060: 0 fractional observations
BPQ070: 0 fractional observations
BPQ090D: 0 fractional observations
BPQ100D: 0 fractional observations

PASS: No genuine decimal variables found in BPQ_I.

--- SEQN ---
Minimum: 83732
Maximum: 93702
Missing: 0
Du